# 02 - Machine Learning Modeling, Anomaly Detection & SHAP Explainability

**MSc Artificial Intelligence & Cybersecurity**

This notebook demonstrates the end-to-end model evaluation, unsupervised anomaly profiling, and SHAP explainability workflows on the CICIDS2017 benchmark.

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib

from src.config import PROCESSED_DIR, CLASSIFICATION_MODEL_DIR, ANOMALY_MODEL_DIR, FIGURES_DIR
from src.explainability.explainer import get_explainer
from src.explainability.risk_engine import RiskAssessmentEngine
print("Environment initialized successfully.")

## 1. Load Preprocessed Benchmark Splits
We load the anti-leakage preprocessed numpy arrays (119,998 train, 30,000 test) across 48 selected features.

In [ ]:
data = np.load(PROCESSED_DIR / "processed_splits.npz")
X_train, X_test = data["X_train"], data["X_test"]
y_train_bin, y_test_bin = data["y_train_bin"], data["y_test_bin"]
y_train_mul, y_test_mul = data["y_train_mul"], data["y_test_mul"]
print(f"X_train shape: {X_train.shape}, X_test shape: {X_test.shape}")

## 2. Load Trained XGBoost Model & Evaluate on Test Split

In [ ]:
xgb_model = joblib.load(CLASSIFICATION_MODEL_DIR / "multiclass_xgboost.joblib")
preds = xgb_model.predict(X_test)
acc = (preds == y_test_mul).mean()
print(f"Multiclass XGBoost Accuracy on Held-Out Test Set: {acc:.4%}")

## 3. Explainable AI with SHAP
Compute TreeExplainer Shapley values for individual packet attribution.

In [ ]:
explainer = get_explainer()
sample_flow = X_test[0:1]
explanation = explainer.explain_instance(sample_flow, top_k=5)
print("Top 5 SHAP contributing features for Flow #0:")
for f in explanation["top_features"]:
    print(f"  - {f['feature']:<25} : SHAP = {f['shap_value']:+.4f} (Raw Value: {f['feature_value']:.2f})")

## 4. Multi-Factor Dynamic Risk Scoring
Evaluate transparent threat risk combining supervised confidence, anomaly detection, port exposure, and CTI.

In [ ]:
risk = RiskAssessmentEngine.evaluate_risk(
    predicted_class="DDoS",
    confidence=0.99,
    is_anomaly=True,
    anomaly_score=0.85,
    destination_port=80,
    cti_reputation=95.0,
    top_shap_features=explanation["top_features"],
)
print(f"Risk Score: {risk['risk_score']}/100 | Tier: {risk['risk_level']}")
print(f"Action: {risk['recommended_action']}")
print(f"Justification: {risk['justification']}")